# Train the square classifier (Colab)

**Pipeline step:** model training. All training logic lives in `scripts/train.py` in the repo; this notebook only moves files and runs it on Colab's GPU.

**Inputs** (upload once to Google Drive, folder `MyDrive/chess_vision/`):
- `crops.zip`: the dataset from `scripts/build_dataset.py` (`train/`, `val/`, `test/`, one folder per class)
- `train.py`: copy of `scripts/train.py`

**Outputs** (written to `MyDrive/chess_vision/outputs/`, so they survive the Colab session ending):
- `model.pt`: best epoch by validation accuracy
- `train_log.csv`: loss and accuracy per epoch

**Before running:** Runtime > Change runtime type > **T4 GPU**.

## 1. Check the GPU
Should print a Tesla T4. If it says no GPU, change the runtime type and rerun.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Mount Google Drive
Gives this session access to `MyDrive`, where the inputs are and the outputs go.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/chess_vision'
!ls -lh {DRIVE}

## 3. Unzip the dataset to local disk
Reading 20k small files straight from Drive is slow; Colab's local disk is fast.

In [ ]:
!mkdir -p /content/data && unzip -q -o {DRIVE}/crops.zip -d /content/data
!for s in train val test; do echo $s $(find /content/data/crops/$s -name '*.jpg' | wc -l); done

## 4. Train
Fine-tunes all layers of pretrained ResNet18 for 10 epochs (Adam, lr 1e-4), light augmentation. Prints one line per epoch; `<- saved` marks a new best validation accuracy. About 5 min per epoch on a T4.

In [ ]:
!python {DRIVE}/train.py --data /content/data/crops --out {DRIVE}/outputs

## 5. Training curves
Train vs. validation loss and accuracy per epoch. A val curve that rises while train keeps falling means overfitting. Figure for the report.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv(f'{DRIVE}/outputs/train_log.csv')
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
log.plot(x='epoch', y=['train_loss', 'val_loss'], ax=ax[0], marker='o', title='Loss')
log.plot(x='epoch', y=['train_acc', 'val_acc'], ax=ax[1], marker='o', title='Accuracy')
plt.tight_layout(); plt.savefig(f'{DRIVE}/outputs/training_curves.png', dpi=150); plt.show()
log

## 6. Done
Download `model.pt` (and the curves) from `MyDrive/chess_vision/outputs/` into `chess_vision/outputs/` on the laptop. The next step, `evaluate.py`, runs on the test split.